# 32 · Select rows by value, checked against pandas `between`

## What this checks

**Select rows by value** keeps the rows whose value lies in a range, from one value to another, both ends included.
This check covers release 0.5.1.

The range was added for a New York workflow on wildfire smoke. There, fourteen days out of 3,653 had to be picked out
by hand, one tick at a time. A range does it in one step.

A range that keeps the wrong rows changes every count after it, without any error. So it is worth checking.

## How it is checked

We use **pandas**, Python's table library, by other authors. Its `between` keeps the values in a range, and it is
what a practitioner would otherwise write. pandas reads the column itself, as dates or as numbers. It does not
compare values the way Fieldwork does.

The data are London's daily deaths from 2002 to 2006. Fieldwork's own reader read them from the published Stata
file, so the dates are the ones the file declares.

Four cases:

- **A range of dates:** the August 2003 heatwave.
- **A range of numbers:** ozone from 60 to 90.
- **An open end:** from Christmas 2006, with no end date.
- **The number range again,** with one cell inside the range emptied.

**The case worth the check is the numbers.** Compared as text, a range of numbers keeps the wrong rows. Text is
compared character by character, so `"100"` sorts before `"60"`. Ozone from 60 to 90 has values on both sides of
every place where the number of digits changes.

## What should happen

1. **Each case keeps exactly the rows pandas keeps:** the same dates, in the same order. The rows kept and dropped
   add up to the table. Rows are compared date by date, so there is no tolerance.
2. **An open end** is compared on one side only: on or after the start.
3. **An emptied cell is outside every range.** pandas never counts an empty value as between. Fieldwork drops that
   row too, so it keeps one row fewer than the ozone case.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_32_select_range import run

report = run(Path.cwd() / "fixtures" / "select-range-london-001.json")
print("agrees:", report["agrees"])
print()
for c in report["comparisons"]:
    print(f"  {'OK ' if c['agrees'] else 'DIFF'} Fieldwork {c['fieldwork']:>4}  pandas {c['pandas']:>4}  {c['note']}")
    print(f"       {c['keyColumn']} from {c['from'] or 'the start'} to {c['to'] or 'the end'}: {c['first']} ... {c['last']}")


## The result on record

First recorded on 4 November 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.5.1:
**Agrees**.

All 4 cases agree, row for row:

- **The August 2003 heatwave:** 15 days.
- **Ozone from 60 to 90, as numbers:** 137 days.
- **From Christmas 2006:** 7 days.
- **Ozone again, with one cell inside the range emptied:** 136 days.

Only the range is checked here. Selecting listed values computes nothing, and Fieldwork's own tests cover it.

Running the cell above repeats the check on the same data.

## The case worth the check: numbers are not text

Compared as text, a range of numbers keeps the wrong rows, and nothing warns you. Both sides read ozone as numbers.
The cell below shows what a text comparison would have kept instead: 193 days, not 137.

In [ ]:
text = report["ozoneComparedAsText"]
print(f"ozone 60 to 90 as numbers : {text['insteadOf']} days")
print(f"ozone 60 to 90 as text    : {text['kept']} days   <- not what either implementation does")


## What this does not show

- **That a range was the right selection.** A period chosen to show a peak is still a choice. Fieldwork's record of
  the run notes it, without judging it.
- **Columns of dates with times.** These dates carry no time. A "to" given as a date leaves out times later that day.

The cell below prints these limits as the check records them.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
